# MergeDeclaredLinks — nightly merges of works linked by a declaration (oxjob #1057, #1099)

Runs between `Locations_with_Types` and `Map_Work_Ids`. When a record declares a link to another record (a preprint to its
article) after both already sit on their own pinned works, MapWorkIds never re-resolves them: this step
merges those works every night. One step for every kind of link (Casey 2026-10-25, oxjob #1057 `MERGE_DECLARED_LINKS.md`):
each **class** has its own pair rule, winner and evidence holds; the holds' shared check, the tripwire, the execute and the
citation moves run once for all classes. It replaces `MergeDeclaredPreprints` (oxjob #1099), whose preprint rule is the
`preprint` class below, cell for cell.

| Class | Pairs | Winner | `merged_work_ids` source | Tripwire |
|---|---|---|---|---|
| `preprint` (oxjob #1099, 2026-10-08) | declared preprint ↔ article DOI pairs, typed (`preprint_link_pairs`) | the group's most-cited work, ties to the older id | `preprint_link_nightly` | 100,000 |
| `version` (oxjob #1696, 2026-11-23) | two Crossref DOIs of one prefix declared versions of one item, or translations of one item within one journal, typed (`crossref_version_link_pairs`) | the group's most-cited work, ties to the older id | `crossref_version_nightly` | 150,000 on its first night (the backlog), then 50,000 |

**preprint** (unchanged from MergeDeclaredPreprints):

MapWorkIds' `crossref_preprint` tier joins a record to its declared
partner only when the link is there on the record's first night; most links are added later (the preprint server records the
article after it is out, and Crossref mirrors the claim onto the article as `has-preprint`), when both records already sit on
their own works and are pinned. This step merges those works every night (Casey 2026-10-08: "build it into the nightly").

- **Pairs** (view `openalex.works.preprint_link_pairs`, recreated here first; MapWorkIds' `crossref_preprint` tier reads its
  Crossref arm): declared by either Crossref record (`is-preprint-of` / `has-preprint` in `ids[]`), or by arXiv's DataCite record
  (`IsVersionOf` a journal DOI; arXiv adds it after publication, oxjob #1256's declared_version pairs); preprint = Crossref
  posted-content/preprint or a DataCite `preprint` (arXiv 10.48550), article = Crossref journal-article typed article / review /
  letter; both pinned, on two works that are in `openalex_works`.
- **Group** = an article's work + the works of every preprint it pairs with. **Winner = the most-cited member, ties to the older
  (lower) id.** A just-arrived article has no citations yet, so it joins the preprint's work (Casey 2026-10-08: the published work
  that comes in and points to a preprint just joins the preprint work); an established, cited article keeps its id (first-night
  backlog: the T2T genome, gnomAD, ARRIVE 2.0 articles would otherwise 404 onto their bioRxiv ids). The merged work presents
  as the article either way (CreateWorksBase prefers the published Crossref VoR, #589).
- **Holds** (evidence of a bad link): the article's work titled as a correction, typed other than article / review / letter, a
  year gap (article > 3 years after or > 1 before the preprint), or content that differs (titles differ and abstract token Jaccard
  < 0.3 or no abstract to compare: blind eval 2026-10-09, 0/100 wrong where content agrees; #1652 2026-10-14, 1/100 wrong
  + 0/15 top-cited in the 0.3-0.6 band, wrong links there are companion papers); arXiv-only:
  the preprint's work already shows another DOI. Mechanical: a preprint work paired with two article works, a work on both sides, a
  loser key held by a work outside the group. Any held pair holds its whole group.

**version** (oxjob #1696; Casey 2026-10-15: Crossref-declared versions are one work, as DataCite's are; 2026-11-16: `is-same-as`
within one prefix and type too). Crossref's ingest keeps only the preprint links in `ids[]`, so this class reads the
declarations from the source table `openalex.crossref.crossref_deduplicated`, as the notice step does.

- **Pairs** (`crossref_version_link_pairs`): `is-version-of` / `has-version` (eLife's Reviewed Preprints and version of
  record, F1000, Research Square, Preprints.org, OSF, Rogue Scholar, the AEA RCT Registry), `is-same-as` (eLife's versionless
  DOI = its current version) and Crossmark `update-to` type `new_version` (Cochrane, F1000); DOI targets of the **same
  prefix** only (two prefixes are two publishers' items: separate works, dedup 2026-10-15 and 2026-11-16). Typed by the
  records' own types: the same type; both article / review / letter / editorial (#1645's article family); or a preprint and
  its published form (2026-10-17); `is-same-as` alone needs the same type. Any other pair of types stays apart (Cochrane
  versions typed `reference-entry` wait for one type, #1722).
- **Translations within one journal** (rel `translation`): `is-translation-of` / `has-translation` between two DOIs of one
  prefix whose records sit on the **same source** (SciELO's per-language DOIs and the like: the same item in another language
  within one journal is one work, dedup 2026-10-15; Jason 2026-11-16). A translation in another journal stays its own work,
  linked by #2536's `translation` link. Same type or the article family only. Bar (pre-registered 2026-11-23): blind check,
  uniform 100: 98 one item, 1 different, 1 unsure; 10 per prefix with >= 150 pairs x 13: 128 one item, 2 different, both
  Global Rheumatology 10.46856 (held whole); 10.26735 held too (a record linked to two different articles).
- **Group** = works connected by pairs (12 rounds of lowest-id propagation). **Winner = the most-cited member, ties to the
  older id.**
- **Holds** (the whole group): `unsettled` (still split after 12 rounds), `group_size` (> 200 works: a bad deposit, not one
  item's versions; largest real group 110), `judged_prefix` (10.63485, Rogue Scholar's Open Access News archive: the
  backlog's blind check found 3 of 20 different items, 2026-11-16), `other_link` (two of the group's works are also joined by
  another declared link, Crossref or a notice: review, part, supplement, citation, retraction, erratum, new edition ...; such
  links mean two works, Jason 2026-11-16, #1490; `crossref_version_other_links`), `loser_key_shared` (as `preprint`).
- **Bar** (oxjob #1696, pre-registered 2026-11-16 before the draw): blind check of the backlog's executable losers, Opus 5.5:
  uniform 300 299 one item, 1 unsure, 0 different; 20 per prefix x 21 prefixes 416 one item, 4 different (10.63485 held
  whole); the 15 most-cited 15/15. First night 2026-11-24 carries the backlog (dry run 2026-11-23: 102,821 losers -> 77,693
  winners; held 3,426 judged_prefix, 2,343 loser_key_shared, 24 other_link, 13 unsettled); a new version is its own work
  until the next night (no MapWorkIds tier yet). With the translation leg (same dry run): 119,129 losers -> 91,942 winners.

**part_of** (oxjob #1057) is off since 2026-10-27 until it is reworked (Casey). It read OpenAlex's `type = 'dataset'`, and
#1647's retyping of DataCite `Collection` / `Other` records widened it from ~2.8K to 273,309 on its first night (tripwire,
nothing written). Its cells are in walden 6e697f63. The 2026-10-23 backlog wave and CreateWorksBase's part rule stay.

- **Shared check:** a work that is a loser or a winner in two classes the same night is held in both (`cross_class`), with its
  whole group, so it is never merged two ways at once.
- **Execute** (as `MergeIdenticalKeyWorks` execute, once for all classes): audit → aliases of the losers' record keys onto the
  winner → drop the losers' map rows → unpin the losers → `merged_work_ids` (the class's source) → ledger. MapWorkIds then
  re-resolves the losers' records onto the winner in this run. Map rows go before pins, so a partial run leaves a loser's records
  pinned where they were, never re-resolving onto it.
- **Citations:** `work_references` rows citing a ledgered loser, and the loser's own reference rows, move to the winner (every
  loser in `declared_link_merges` and the old `preprint_link_merges` history).
- **Tripwire:** more executable losers than the class's cap fails the task before any write (a bug, not a backlog).
  `Merge_Declared_Links_Gate` passes either way, so MapWorkIds always runs.
- **Undo** (before the next nightly), per class (`version`: `link = 'version'`, source `crossref_version_nightly`): re-insert the run's `declared_link_merges_audit` pin / map rows of that `link`,
  delete its alias rows from `work_id_map`, delete the run's `merged_work_ids` rows (that class's source). The preprint step's
  own tables (`preprint_link_merges`, `_audit`, `_refs_audit`) stay as history; nothing writes them after this replaced it.

In [ ]:
CREATE OR REPLACE VIEW openalex.works.preprint_link_pairs
COMMENT 'oxjob #1099: declared preprint <-> published-article DOI pairs, typed. One definition for MapWorkIds (crossref_preprint tier, arm = crossref) and MergeDeclaredPreprints (both arms). Recreated nightly by MergeDeclaredPreprints.'
AS
-- Crossref: declared by either record (`is-preprint-of` on the preprint, `has-preprint` on the article; Crossref mirrors one
-- side's claim onto the other)
WITH cr_links AS (
  SELECT DISTINCT
         CASE WHEN x.relationship = 'is-preprint-of' THEN regexp_replace(l.native_id, '[^a-zA-Z0-9\./-]', '') ELSE regexp_replace(x.id, '[^a-zA-Z0-9\./-]', '') END AS pre,
         CASE WHEN x.relationship = 'is-preprint-of' THEN regexp_replace(x.id, '[^a-zA-Z0-9\./-]', '') ELSE regexp_replace(l.native_id, '[^a-zA-Z0-9\./-]', '') END AS pub
  FROM openalex.works.locations_w_types l LATERAL VIEW explode(l.ids) e AS x
  WHERE l.provenance = 'crossref' AND EXISTS(l.ids, y -> y.relationship IN ('is-preprint-of', 'has-preprint'))
    AND x.relationship IN ('is-preprint-of', 'has-preprint')
),
-- arXiv's DataCite record declaring IsVersionOf its journal DOI (oxjob #1256's declared_version pairs); arXiv adds the DOI
-- after the article is out, so most of these appear when both records are already pinned
dc_links AS (
  SELECT DISTINCT regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '') AS pre,
         regexp_replace(regexp_replace(LOWER(x.id), '^https?://(dx\\.)?doi\\.org/', ''), '[^a-zA-Z0-9\./-]', '') AS pub
  FROM openalex.works.locations_w_types l LATERAL VIEW explode(l.ids) e AS x
  WHERE l.provenance = 'datacite' AND l.type = 'preprint' AND l.native_id LIKE '10.48550/%'
    AND x.relationship = 'IsVersionOf' AND LOWER(COALESCE(x.namespace, 'doi')) = 'doi' AND LOWER(x.id) NOT LIKE '%10.48550/%'
),
links AS (
  SELECT pre, pub, CASE WHEN BOOL_OR(arm = 'crossref') THEN 'crossref' ELSE 'datacite' END AS arm
  FROM (SELECT pre, pub, 'crossref' AS arm FROM cr_links UNION ALL SELECT pre, pub, 'datacite' FROM dc_links)
  WHERE pre <> pub
  GROUP BY pre, pub
)
-- typed: preprint = Crossref posted-content/preprint or a DataCite preprint (AACR registers ~204K Figshare supplements as
-- `is-preprint-of` with subtype `other`); article = Crossref journal-article typed article / review / letter
SELECT k.pre, k.pub, k.arm FROM links k
LEFT SEMI JOIN openalex.works.locations_w_types a
  ON a.provenance IN ('crossref', 'datacite') AND regexp_replace(LOWER(a.native_id), '[^a-zA-Z0-9\./-]', '') = k.pre
 AND ((a.provenance = 'crossref' AND a.raw_type = 'posted-content' AND a.type = 'preprint')
   OR (a.provenance = 'datacite' AND a.type = 'preprint'))
LEFT SEMI JOIN openalex.works.locations_w_types b
  ON b.provenance = 'crossref' AND regexp_replace(b.native_id, '[^a-zA-Z0-9\./-]', '') = k.pub
 AND b.raw_type = 'journal-article' AND b.type IN ('article', 'review', 'letter')

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.declared_link_merges (
  run_ts TIMESTAMP, link STRING, loser_work_id BIGINT, winner_work_id BIGINT, group_work_id BIGINT, n_pairs INT, signals STRING)
CLUSTER BY (loser_work_id)

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.declared_link_merges_audit (
  run_ts TIMESTAMP, link STRING, kind STRING, loser_work_id BIGINT, winner_work_id BIGINT,
  provenance STRING, native_id_namespace STRING, native_id STRING, work_id_source STRING,
  openalex_created_dt DATE, openalex_updated_dt TIMESTAMP, seeded_dt DATE, seeded_from STRING,
  doi STRING, pmid STRING, arxiv STRING, title_author STRING, created_date DATE, updated_date TIMESTAMP)
CLUSTER BY (run_ts, loser_work_id)

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.declared_link_merges_refs_audit (
  run_ts TIMESTAMP, side STRING, citing_work_id BIGINT, native_id STRING, native_id_namespace STRING, ref_ind BIGINT,
  old_id BIGINT, new_id BIGINT)

## Class `preprint` (oxjob #1099): pairs on two works, evidence holds, groups and winners
The MergeDeclaredPreprints cells, unchanged.

In [ ]:
CREATE OR REPLACE TABLE openalex.works.preprint_link_merge_pairs AS
WITH links AS (SELECT pre, pub, arm = 'crossref' AS cr_declared FROM openalex.works.preprint_link_pairs),
ld AS (SELECT pre AS d FROM links UNION SELECT pub FROM links),
lt AS (
  SELECT regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '') AS d, l.provenance, r.work_id
  FROM openalex.works.locations_w_types l
  LEFT SEMI JOIN ld ON ld.d = regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '')
  JOIN openalex.works.location_work_ids r
    ON r.provenance = l.provenance AND r.native_id_namespace = l.native_id_namespace AND r.native_id = l.native_id
  WHERE l.provenance IN ('crossref', 'datacite') AND r.work_id IS NOT NULL
),
pw AS (
  SELECT DISTINCT k.pre AS pre_doi, k.pub AS pub_doi, k.cr_declared, a.work_id AS pre_work, b.work_id AS pub_work
  FROM links k JOIN lt a ON a.d = k.pre JOIN lt b ON b.d = k.pub AND b.provenance = 'crossref'
  WHERE a.work_id <> b.work_id
),
wp AS (SELECT DISTINCT pre_work, pub_work FROM pw),
ids AS (SELECT pre_work AS id FROM wp UNION SELECT pub_work FROM wp),
wsel AS (
  SELECT w.id, w.publication_year AS yr, w.type, w.title, COALESCE(w.cited_by_count, 0) AS cites, w.abstract,
         regexp_replace(lower(w.title), '[^\\p{L}\\p{N}]', '') AS tn,
         regexp_replace(lower(w.doi), '^https?://(dx\\.)?doi\\.org/', '') AS doi
  FROM openalex.works.openalex_works w LEFT SEMI JOIN ids ON ids.id = w.id
),
-- abstract tokens as rows, Jaccard by join: array set functions over per-work token lists gave run-varying answers in both
-- directions on the warehouse (2026-10-07 arrays_overlap; 2026-10-09 size(array_intersect): 61-193 pairs flipped per run)
atok AS (
  SELECT DISTINCT id, t FROM wsel
  LATERAL VIEW explode(split(regexp_replace(lower(COALESCE(abstract, '')), '[^a-z0-9 ]', ' '), ' +')) e AS t
  WHERE length(t) > 3
),
nab AS (SELECT id, COUNT(*) AS n FROM atok GROUP BY id),
ain AS (SELECT p.pre_work, p.pub_work, COUNT(*) AS inter FROM wp p JOIN atok a ON a.id = p.pre_work JOIN atok b ON b.id = p.pub_work AND b.t = a.t GROUP BY 1, 2),
wf AS (
  SELECT s.id, s.yr, s.type, s.title, s.cites, s.tn, s.doi, COALESCE(na.n, 0) AS n_abs
  FROM wsel s LEFT JOIN nab na ON na.id = s.id
),
r AS (
  SELECT p.*, x.yr AS pre_yr, y.yr AS pub_yr, y.type AS pub_type, x.cites AS pre_cites, y.cites AS pub_cites,
         LEFT(x.title, 200) AS pre_title, LEFT(y.title, 200) AS pub_title, y.title AS pub_title_full,
         -- oxjob #1676: a missing or empty title never agrees and never skips the content_differs hold. Plain x.tn = y.tn was
         -- NULL for an untitled side (NOT NULL is NULL, so the hold was skipped and the pair merged unchecked) and TRUE for two ''
         COALESCE(x.tn = y.tn AND x.tn <> '', FALSE) AS title_same,
         x.doi AS pre_work_doi,
         CASE WHEN x.n_abs >= 20 AND y.n_abs >= 20 THEN COALESCE(ai.inter, 0) / (x.n_abs + y.n_abs - COALESCE(ai.inter, 0)) END AS abs_jaccard
  FROM pw p JOIN wf x ON x.id = p.pre_work JOIN wf y ON y.id = p.pub_work
  LEFT JOIN ain ai ON ai.pre_work = p.pre_work AND ai.pub_work = p.pub_work
)
SELECT pre_doi, pub_doi, pre_work, pub_work, pre_yr, pub_yr, pub_type, pre_cites, pub_cites, pre_title, pub_title,
       CASE WHEN cr_declared THEN 'crossref' ELSE 'datacite' END AS arm, abs_jaccard,
       concat_ws('+', 'declared', CASE WHEN title_same THEN 'title' END, CASE WHEN abs_jaccard >= 0.3 THEN 'abstract' END) AS signals,
       CASE WHEN pub_title_full RLIKE '(?i)^\\s*(metadata correction|correction|erratum|corrigendum|addendum)([^a-z]|$)' THEN 'correction_notice'
            WHEN pub_type NOT IN ('article', 'review', 'letter') THEN 'article_type'
            WHEN pre_yr IS NOT NULL AND pub_yr IS NOT NULL AND (pub_yr - pre_yr > 3 OR pub_yr - pre_yr < -1) THEN 'year_gap'
            -- the declaration merges when the content agrees: same normalized title, or abstracts agree (token Jaccard >= 0.3). Blind
            -- eval 2026-10-09 (240 pairs): 0/100 wrong where content agrees; every wrong link (JMIR's 2018 off-by-N deposits, OSF
            -- REMOVED/WITHDRAWN placeholders, mistyped DOIs) had a different title and abstracts < 0.6 or none. The 0.3-0.6 band
            -- (#1652, 2026-10-14): 1/100 wrong + 0/15 top-cited; its wrong links are companion papers (0.33, 0.42). An author-name check
            -- let ~0.7 % wrong through (common surnames, same-group papers) and held collaboration papers ("ATLAS Collaboration").
            WHEN NOT title_same AND COALESCE(abs_jaccard, 0) < 0.3 THEN 'content_differs'
            -- arXiv-only: the preprint's work already presents another DOI (a journal record glued on): a three-way case
            WHEN NOT cr_declared AND pre_work_doi IS NOT NULL AND pre_work_doi <> pre_doi AND pre_work_doi NOT LIKE '10.48550/%' THEN 'pre_work_other_doi'
       END AS evidence_hold
FROM r

In [ ]:
CREATE OR REPLACE TABLE openalex.works.preprint_link_merge_losers AS
WITH multi AS (SELECT pre_work FROM openalex.works.preprint_link_merge_pairs GROUP BY pre_work HAVING COUNT(DISTINCT pub_work) > 1),
both_sides AS (SELECT DISTINCT a.pub_work AS w FROM openalex.works.preprint_link_merge_pairs a
               JOIN openalex.works.preprint_link_merge_pairs b ON b.pre_work = a.pub_work),
pr AS (
  SELECT p.*,
         COALESCE(p.evidence_hold,
                  CASE WHEN m.pre_work IS NOT NULL THEN 'multi_article'
                       WHEN c1.w IS NOT NULL OR c2.w IS NOT NULL THEN 'chained' END) AS pair_hold
  FROM openalex.works.preprint_link_merge_pairs p
  LEFT JOIN multi m ON m.pre_work = p.pre_work
  LEFT JOIN both_sides c1 ON c1.w = p.pub_work
  LEFT JOIN both_sides c2 ON c2.w = p.pre_work
),
mc AS (SELECT pub_work AS g, pre_work AS w, pre_cites AS cites FROM pr UNION SELECT pub_work, pub_work, pub_cites FROM pr),
grp AS (
  SELECT p.pub_work AS group_work_id, w.winner_work_id, COUNT(*) AS n_pairs, MIN(p.pair_hold) AS group_hold, MIN(p.signals) AS signals
  FROM pr p
  JOIN (SELECT g, MAX_BY(w, named_struct('cites', cites, 'older', -w)) AS winner_work_id FROM mc GROUP BY g) w ON w.g = p.pub_work
  GROUP BY p.pub_work, w.winner_work_id
),
members AS (
  SELECT DISTINCT g.group_work_id, g.winner_work_id, m.w AS work_id
  FROM grp g JOIN mc m ON m.g = g.group_work_id
),
losers AS (
  SELECT m.group_work_id, m.winner_work_id, m.work_id AS loser_work_id, g.n_pairs, g.signals, g.group_hold
  FROM members m JOIN grp g ON g.group_work_id = m.group_work_id
  WHERE m.work_id <> m.winner_work_id
),
-- the losers' record keys, as MapWorkIds will look them up after the unpin
lrec AS (
  SELECT lo.group_work_id, lo.loser_work_id, l.merge_key AS mk
  FROM losers lo
  JOIN openalex.works.location_work_ids r ON r.work_id = lo.loser_work_id
  JOIN openalex.works.locations_w_types l
    ON l.provenance = r.provenance AND l.native_id_namespace = r.native_id_namespace AND l.native_id = r.native_id
  WHERE lo.group_hold IS NULL
),
lkeys AS (
  SELECT DISTINCT x.group_work_id, x.loser_work_id, kv.col, kv.k
  FROM lrec x LATERAL VIEW explode(map('doi', NULLIF(x.mk.doi, ''), 'pmid', NULLIF(x.mk.pmid, ''),
                                       'arxiv', CASE WHEN x.mk.arxiv IN ('arXiv:', '') THEN NULL ELSE x.mk.arxiv END,
                                       'title_author', NULLIF(x.mk.title_author, ''))) kv AS col, k
  WHERE kv.k IS NOT NULL
),
keyless AS (
  SELECT DISTINCT group_work_id, loser_work_id FROM lrec
  WHERE NULLIF(mk.doi, '') IS NULL AND NULLIF(mk.pmid, '') IS NULL AND (mk.arxiv IS NULL OR mk.arxiv IN ('arXiv:', ''))
),
kin AS (SELECT DISTINCT col, k FROM lkeys),
holders AS (
  SELECT 'doi' AS col, m.doi AS k, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'doi' AND kin.k = m.doi
  UNION ALL SELECT 'pmid', m.pmid, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'pmid' AND kin.k = m.pmid
  UNION ALL SELECT 'arxiv', m.arxiv, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'arxiv' AND kin.k = m.arxiv
  UNION ALL SELECT 'title_author', m.title_author, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'title_author' AND kin.k = m.title_author
),
-- a key held by a work outside the group; a shared title key only counts for a loser with a doi / pmid / arxiv-less record
shared AS (
  SELECT DISTINCT lk.group_work_id FROM lkeys lk JOIN holders h ON h.col = lk.col AND h.k = lk.k
  WHERE NOT EXISTS (SELECT 1 FROM members m WHERE m.group_work_id = lk.group_work_id AND m.work_id = h.id)
    AND (lk.col <> 'title_author' OR EXISTS (SELECT 1 FROM keyless kl WHERE kl.group_work_id = lk.group_work_id AND kl.loser_work_id = lk.loser_work_id))
)
SELECT lo.group_work_id, lo.winner_work_id, lo.loser_work_id, lo.n_pairs, lo.signals,
       COALESCE(lo.group_hold, CASE WHEN sh.group_work_id IS NOT NULL THEN 'loser_key_shared' END) AS hold_reason,
       current_timestamp() AS run_ts
FROM losers lo LEFT JOIN shared sh ON sh.group_work_id = lo.group_work_id

In [ ]:
SELECT hold_reason, COUNT(*) AS losers, COUNT(DISTINCT winner_work_id) AS winners
FROM openalex.works.preprint_link_merge_losers GROUP BY 1 ORDER BY 2 DESC

## Class `version` (oxjob #1696): Crossref-declared versions (and one journal's language editions) on two works, typed, groups and winners
Pairs from Crossref's own declarations, a guard table of other links between them, pairs on two works, connected
groups, then losers with their holds.

In [ ]:
CREATE OR REPLACE TABLE openalex.works.crossref_version_link_pairs
COMMENT 'oxjob #1696: two Crossref DOIs of one prefix that the publisher declares versions of one item (or two language editions of one item in one journal), typed. Undirected (x < y), cleaned like doi_clean. Read by MergeDeclaredLinks (version class). Rebuilt nightly by MergeDeclaredLinks.'
AS
-- Crossref's own declarations, read from the source table (one row per record): `is-version-of` / `has-version` (eLife,
-- Preprints.org, Research Square, OSF, Rogue Scholar, the AEA RCT Registry), `is-same-as` (eLife's versionless DOI = its current
-- version) and Crossmark `update-to` type `new_version` (Cochrane, F1000). The ingest's ids[] carries only the preprint links.
-- Also `is-translation-of` / `has-translation` within one journal (SciELO's per-language DOIs: one work, dedup 2026-10-15; a
-- translation in another journal is its own work with a translation link, #2536).
WITH rel AS (
  SELECT lower(native_id) AS doi, relation, `update-to` AS upd
  FROM openalex.crossref.crossref_deduplicated
  WHERE relation.`is-version-of` IS NOT NULL OR relation.`has-version` IS NOT NULL OR relation.`is-same-as` IS NOT NULL
     OR exists(`update-to`, u -> lower(u.type) = 'new_version')
     OR relation.`is-translation-of` IS NOT NULL OR relation.`has-translation` IS NOT NULL
),
e AS (
  SELECT doi AS a, r.id AS b, 'is-version-of' AS rel FROM rel LATERAL VIEW explode(relation.`is-version-of`) t AS r WHERE r.`id-type` = 'doi'
  UNION ALL
  SELECT doi, r.id, 'has-version' FROM rel LATERAL VIEW explode(relation.`has-version`) t AS r WHERE r.`id-type` = 'doi'
  UNION ALL
  SELECT doi, r.id, 'is-same-as' FROM rel LATERAL VIEW explode(relation.`is-same-as`) t AS r WHERE r.`id-type` = 'doi'
  UNION ALL
  SELECT doi, u.DOI, 'new_version' FROM rel LATERAL VIEW explode(upd) t AS u WHERE lower(u.type) = 'new_version'
  UNION ALL
  SELECT doi, r.id, 'translation' FROM rel LATERAL VIEW explode(relation.`is-translation-of`) t AS r WHERE r.`id-type` = 'doi'
  UNION ALL
  SELECT doi, r.id, 'translation' FROM rel LATERAL VIEW explode(relation.`has-translation`) t AS r WHERE r.`id-type` = 'doi'
),
c AS (
  SELECT regexp_replace(a, '[^a-zA-Z0-9\./-]', '') AS a,
         regexp_replace(regexp_replace(lower(trim(b)), '^https?://(dx\\.)?doi\\.org/', ''), '[^a-zA-Z0-9\./-]', '') AS b, rel
  FROM e
),
-- one prefix only: two prefixes are two publishers' items, which stay separate works (dedup § 5, 2026-10-15 and 2026-11-16)
und AS (
  SELECT least(a, b) AS x, greatest(a, b) AS y, array_sort(collect_set(rel)) AS rels
  FROM c
  WHERE a <> b AND b RLIKE '^10[.][0-9]{4,}/.+' AND split_part(a, '/', 1) = split_part(b, '/', 1)
  GROUP BY 1, 2
),
ds AS (SELECT x AS d FROM und UNION SELECT y FROM und),
-- the records' own types (matching trusts the record's type, 2026-10-15)
rt AS (
  SELECT regexp_replace(lower(l.native_id), '[^a-zA-Z0-9\./-]', '') AS d, MIN(l.type) AS type,
         -- the record's journal, for translations (one source only)
         CASE WHEN MIN(l.source_id) = MAX(l.source_id) THEN MIN(l.source_id) END AS source_id
  FROM openalex.works.locations_w_types l
  LEFT SEMI JOIN ds ON ds.d = regexp_replace(lower(l.native_id), '[^a-zA-Z0-9\./-]', '')
  WHERE l.provenance = 'crossref'
  GROUP BY 1
),
typed AS (
  SELECT u.x, u.y, u.rels, a.type AS x_type, b.type AS y_type,
         -- a translation alone joins only within one journal (same source), never as a preprint of its original
         CASE WHEN u.rels = array('translation') AND NOT COALESCE(a.source_id = b.source_id, FALSE) THEN NULL
              -- prefixes whose translation links the leg's blind check found joining different items (2026-11-23): Global
              -- Rheumatology 10.46856 2 of 10; 10.26735 (Naif Arab University) a record linked to two different articles
              WHEN u.rels = array('translation') AND split_part(u.x, '/', 1) IN ('10.46856', '10.26735') THEN NULL
              WHEN a.type = b.type THEN 'same_type'
              -- is-same-as alone joins only the same type (Casey 2026-11-16)
              WHEN u.rels = array('is-same-as') THEN NULL
              -- the article family, as #1645's waves use it
              WHEN a.type IN ('article', 'review', 'letter', 'editorial') AND b.type IN ('article', 'review', 'letter', 'editorial')
              THEN 'article_family'
              -- a translation is never a preprint of its original
              WHEN u.rels = array('translation') THEN NULL
              -- a preprint joins its published version, whatever form that takes (2026-10-17)
              WHEN (a.type = 'preprint' AND b.type IN ('article', 'review', 'letter', 'editorial', 'book', 'book-chapter',
                                                       'conference-paper', 'report', 'dissertation'))
                OR (b.type = 'preprint' AND a.type IN ('article', 'review', 'letter', 'editorial', 'book', 'book-chapter',
                                                       'conference-paper', 'report', 'dissertation'))
              THEN 'preprint_to_published' END AS type_rule
  FROM und u JOIN rt a ON a.d = u.x JOIN rt b ON b.d = u.y
)
-- any other pair of types stays apart: a declaration never merges different items (2026-10-15), e.g. ASME's journal article
-- declared a new version of its Turbo Expo paper; Cochrane's versions typed reference-entry wait for one type (#1722)
SELECT x, y, rels, x_type, y_type, type_rule FROM typed WHERE type_rule IS NOT NULL

In [ ]:
CREATE OR REPLACE TABLE openalex.works.crossref_version_other_links
COMMENT 'oxjob #1696: two DOIs of the version class that are also joined by another declared link (review, part, supplement, comment, a notice, a new edition ...), on their works. Such links mean two works (Jason 2026-11-16, #1490), so the class holds a group holding both ends. Rebuilt nightly by MergeDeclaredLinks.'
AS
WITH ds AS (
  SELECT x AS d FROM openalex.works.crossref_version_link_pairs UNION SELECT y FROM openalex.works.crossref_version_link_pairs
),
rec AS (
  SELECT regexp_replace(lower(c.native_id), '[^a-zA-Z0-9\./-]', '') AS a, c.relation, c.`update-to` AS upd
  FROM openalex.crossref.crossref_deduplicated c
  LEFT SEMI JOIN ds ON ds.d = regexp_replace(lower(c.native_id), '[^a-zA-Z0-9\./-]', '')
),
-- every Crossref relation of these records but its intra-work ones (Crossref relations.xsd: version, same-as, identical,
-- preprint, manuscript, replaces, format, expression, manifestation, variant form, translation; one journal's language
-- versions are one work, dedup 2026-10-15); every Crossmark update but new_version
o AS (
  SELECT r.a, m.k AS rel, x.id AS b
  FROM rec r
  LATERAL VIEW explode(from_json(to_json(r.relation), 'map<string, array<struct<id: string, `id-type`: string>>>')) m AS k, v
  LATERAL VIEW explode(m.v) t AS x
  WHERE x.`id-type` = 'doi'
    AND m.k NOT IN ('is-version-of', 'has-version', 'is-same-as', 'is-identical-to', 'is-preprint-of', 'has-preprint',
                    'is-manuscript-of', 'has-manuscript', 'is-replaced-by', 'replaces', 'is-format-of', 'has-format',
                    'is-expression-of', 'has-expression', 'is-manifestation-of', 'has-manifestation', 'is-variant-form-of',
                    'is-original-form-of', 'is-translation-of', 'has-translation')
  UNION ALL
  SELECT r.a, concat('update-to:', lower(u.type)), u.DOI
  FROM rec r LATERAL VIEW explode(r.upd) t AS u
  WHERE lower(u.type) <> 'new_version'
  -- notices (retraction, correction, comment ...) from every source the notice step reads (oxjob #2169)
  UNION ALL
  SELECT n.notice_id, concat('notice:', n.notice_type), n.target_id
  FROM openalex.works.notice_link_pairs n
  WHERE n.notice_ns = 'doi' AND n.target_ns = 'doi'
    AND (regexp_replace(lower(n.notice_id), '[^a-zA-Z0-9\./-]', '') IN (SELECT d FROM ds)
      OR regexp_replace(lower(n.target_id), '[^a-zA-Z0-9\./-]', '') IN (SELECT d FROM ds))
),
c AS (
  SELECT regexp_replace(lower(a), '[^a-zA-Z0-9\./-]', '') AS a,
         regexp_replace(regexp_replace(lower(trim(b)), '^https?://(dx\\.)?doi\\.org/', ''), '[^a-zA-Z0-9\./-]', '') AS b, rel
  FROM o
),
und AS (SELECT least(a, b) AS x, greatest(a, b) AS y, array_sort(collect_set(rel)) AS rels FROM c WHERE a <> b GROUP BY 1, 2),
ods AS (SELECT x AS d FROM und UNION SELECT y FROM und),
lt AS (
  SELECT regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '') AS d, MIN(r.work_id) AS work_id
  FROM openalex.works.locations_w_types l
  LEFT SEMI JOIN ods ON ods.d = regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '')
  JOIN openalex.works.location_work_ids r
    ON r.provenance = l.provenance AND r.native_id_namespace = l.native_id_namespace AND r.native_id = l.native_id
  WHERE l.provenance = 'crossref' AND r.work_id IS NOT NULL
  GROUP BY 1
)
SELECT u.x, u.y, u.rels, a.work_id AS x_work, b.work_id AS y_work
FROM und u JOIN lt a ON a.d = u.x JOIN lt b ON b.d = u.y
WHERE a.work_id <> b.work_id

In [ ]:
CREATE OR REPLACE TABLE openalex.works.crossref_version_merge_pairs AS
WITH k AS (SELECT x, y, rels, type_rule FROM openalex.works.crossref_version_link_pairs),
ds AS (SELECT x AS d FROM k UNION SELECT y FROM k),
lt AS (
  SELECT regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '') AS d, MIN(r.work_id) AS work_id
  FROM openalex.works.locations_w_types l
  LEFT SEMI JOIN ds ON ds.d = regexp_replace(LOWER(l.native_id), '[^a-zA-Z0-9\./-]', '')
  JOIN openalex.works.location_work_ids r
    ON r.provenance = l.provenance AND r.native_id_namespace = l.native_id_namespace AND r.native_id = l.native_id
  WHERE l.provenance = 'crossref' AND r.work_id IS NOT NULL
  GROUP BY 1
)
SELECT k.x, k.y, k.rels, k.type_rule, a.work_id AS x_work, b.work_id AS y_work
FROM k JOIN lt a ON a.d = k.x JOIN lt b ON b.d = k.y
WHERE a.work_id <> b.work_id

In [ ]:
CREATE OR REPLACE TABLE openalex.works.crossref_version_groups AS
-- connected groups of works joined by version pairs: each round, every work takes the lowest group id among itself and its
-- neighbours (each work is its own neighbour); after 12 rounds a group still split across an edge is held as unsettled
WITH e AS (
  SELECT x_work AS a, y_work AS b FROM openalex.works.crossref_version_merge_pairs
  UNION SELECT y_work, x_work FROM openalex.works.crossref_version_merge_pairs
  UNION SELECT x_work, x_work FROM openalex.works.crossref_version_merge_pairs
  UNION SELECT y_work, y_work FROM openalex.works.crossref_version_merge_pairs
),
l0 AS (SELECT a AS w, MIN(b) AS g FROM e GROUP BY a),
l1 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l0 n ON n.w = e.b GROUP BY e.a),
l2 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l1 n ON n.w = e.b GROUP BY e.a),
l3 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l2 n ON n.w = e.b GROUP BY e.a),
l4 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l3 n ON n.w = e.b GROUP BY e.a),
l5 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l4 n ON n.w = e.b GROUP BY e.a),
l6 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l5 n ON n.w = e.b GROUP BY e.a),
l7 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l6 n ON n.w = e.b GROUP BY e.a),
l8 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l7 n ON n.w = e.b GROUP BY e.a),
l9 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l8 n ON n.w = e.b GROUP BY e.a),
l10 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l9 n ON n.w = e.b GROUP BY e.a),
l11 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l10 n ON n.w = e.b GROUP BY e.a),
l12 AS (SELECT e.a AS w, MIN(n.g) AS g FROM e JOIN l11 n ON n.w = e.b GROUP BY e.a)
SELECT w AS work_id, g AS group_work_id FROM l12

In [ ]:
CREATE OR REPLACE TABLE openalex.works.crossref_version_merge_losers AS
WITH gr AS (SELECT work_id, group_work_id FROM openalex.works.crossref_version_groups),
-- a group holding both ends of another declared link (review, part, supplement, comment, notice, new edition ...): those are
-- two works (Jason 2026-11-16, #1490), so the group waits for a person
other_link AS (
  SELECT DISTINCT gx.group_work_id AS g
  FROM openalex.works.crossref_version_other_links o
  JOIN gr gx ON gx.work_id = o.x_work JOIN gr gy ON gy.work_id = o.y_work
  WHERE gx.group_work_id = gy.group_work_id
),
unsettled AS (
  SELECT DISTINCT x.group_work_id AS g
  FROM openalex.works.crossref_version_merge_pairs p
  JOIN gr x ON x.work_id = p.x_work JOIN gr y ON y.work_id = p.y_work
  WHERE x.group_work_id <> y.group_work_id
  UNION
  SELECT DISTINCT y.group_work_id
  FROM openalex.works.crossref_version_merge_pairs p
  JOIN gr x ON x.work_id = p.x_work JOIN gr y ON y.work_id = p.y_work
  WHERE x.group_work_id <> y.group_work_id
),
mem AS (
  SELECT l.group_work_id, l.work_id, COALESCE(w.cited_by_count, 0) AS cites
  FROM gr l LEFT JOIN openalex.works.openalex_works w ON w.id = l.work_id
),
grp AS (
  SELECT m.group_work_id,
         -- the most-cited member, ties to the older (lower) id, as the preprint class
         MAX_BY(m.work_id, named_struct('cites', m.cites, 'older', -m.work_id)) AS winner_work_id,
         COUNT(*) AS n_works
  FROM mem m GROUP BY m.group_work_id
),
sig AS (
  SELECT l.group_work_id, COUNT(*) AS n_pairs, MIN(split_part(p.x, '/', 1)) AS prefix,
         concat_ws('+', 'declared', array_join(array_sort(collect_set(p.type_rule)), '+'),
                   array_join(array_sort(array_distinct(flatten(collect_list(p.rels)))), '+')) AS signals
  FROM openalex.works.crossref_version_merge_pairs p JOIN gr l ON l.work_id = p.x_work
  GROUP BY l.group_work_id
),
losers AS (
  SELECT m.group_work_id, g.winner_work_id, m.work_id AS loser_work_id, s.n_pairs, s.signals,
         CASE WHEN u.g IS NOT NULL THEN 'unsettled'
              WHEN ol.g IS NOT NULL THEN 'other_link'
              -- a group this big is a bad deposit linking unrelated items, not one item's versions (largest real one
              -- 2026-11-16: 110, a Preprints.org preprint's 110 versions)
              WHEN g.n_works > 200 THEN 'group_size'
              -- a prefix whose declarations the backlog's blind check found joining different items (3 of 20, 2026-11-16):
              -- Rogue Scholar's archive of the Open Access News blog links same-titled posts as versions
              WHEN s.prefix IN ('10.63485') THEN 'judged_prefix' END AS group_hold
  FROM mem m JOIN grp g ON g.group_work_id = m.group_work_id
  LEFT JOIN sig s ON s.group_work_id = m.group_work_id
  LEFT JOIN unsettled u ON u.g = m.group_work_id
  LEFT JOIN other_link ol ON ol.g = m.group_work_id
  WHERE m.work_id <> g.winner_work_id
),
-- the losers' record keys, as MapWorkIds will look them up after the unpin (the preprint class's check)
lrec AS (
  SELECT lo.group_work_id, lo.loser_work_id, l.merge_key AS mk
  FROM losers lo
  JOIN openalex.works.location_work_ids r ON r.work_id = lo.loser_work_id
  JOIN openalex.works.locations_w_types l
    ON l.provenance = r.provenance AND l.native_id_namespace = r.native_id_namespace AND l.native_id = r.native_id
  WHERE lo.group_hold IS NULL
),
lkeys AS (
  SELECT DISTINCT x.group_work_id, x.loser_work_id, kv.col, kv.k
  FROM lrec x LATERAL VIEW explode(map('doi', NULLIF(x.mk.doi, ''), 'pmid', NULLIF(x.mk.pmid, ''),
                                       'arxiv', CASE WHEN x.mk.arxiv IN ('arXiv:', '') THEN NULL ELSE x.mk.arxiv END,
                                       'title_author', NULLIF(x.mk.title_author, ''))) kv AS col, k
  WHERE kv.k IS NOT NULL
),
keyless AS (
  SELECT DISTINCT group_work_id, loser_work_id FROM lrec
  WHERE NULLIF(mk.doi, '') IS NULL AND NULLIF(mk.pmid, '') IS NULL AND (mk.arxiv IS NULL OR mk.arxiv IN ('arXiv:', ''))
),
kin AS (SELECT DISTINCT col, k FROM lkeys),
holders AS (
  SELECT 'doi' AS col, m.doi AS k, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'doi' AND kin.k = m.doi
  UNION ALL SELECT 'pmid', m.pmid, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'pmid' AND kin.k = m.pmid
  UNION ALL SELECT 'arxiv', m.arxiv, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'arxiv' AND kin.k = m.arxiv
  UNION ALL SELECT 'title_author', m.title_author, m.id FROM openalex.works.work_id_map m JOIN kin ON kin.col = 'title_author' AND kin.k = m.title_author
),
-- a key held by a work outside the group; a shared title key only counts for a loser with a doi / pmid / arxiv-less record
shared AS (
  SELECT DISTINCT lk.group_work_id FROM lkeys lk JOIN holders h ON h.col = lk.col AND h.k = lk.k
  WHERE NOT EXISTS (SELECT 1 FROM mem m WHERE m.group_work_id = lk.group_work_id AND m.work_id = h.id)
    AND (lk.col <> 'title_author' OR EXISTS (SELECT 1 FROM keyless kl WHERE kl.group_work_id = lk.group_work_id AND kl.loser_work_id = lk.loser_work_id))
)
SELECT lo.group_work_id, lo.winner_work_id, lo.loser_work_id, lo.n_pairs, lo.signals,
       COALESCE(lo.group_hold, CASE WHEN sh.group_work_id IS NOT NULL THEN 'loser_key_shared' END) AS hold_reason,
       current_timestamp() AS run_ts
FROM losers lo LEFT JOIN shared sh ON sh.group_work_id = lo.group_work_id

In [ ]:
SELECT hold_reason, COUNT(*) AS losers, COUNT(DISTINCT winner_work_id) AS winners
FROM openalex.works.crossref_version_merge_losers GROUP BY 1 ORDER BY 2 DESC

## All classes: one row per loser, the cross-class hold

In [ ]:
CREATE OR REPLACE TABLE openalex.works.declared_link_losers AS
WITH u AS (
  SELECT 'preprint' AS link, group_work_id, winner_work_id, loser_work_id, n_pairs, signals, hold_reason, run_ts
  FROM openalex.works.preprint_link_merge_losers
  UNION ALL
  SELECT 'version', group_work_id, winner_work_id, loser_work_id, n_pairs, signals, hold_reason, run_ts
  FROM openalex.works.crossref_version_merge_losers
),
ex AS (SELECT link, winner_work_id, loser_work_id FROM u WHERE hold_reason IS NULL),
-- a work that is a loser or a winner in two classes' executable rows the same night
cc AS (
  SELECT id FROM (SELECT link, winner_work_id AS id FROM ex UNION SELECT link, loser_work_id FROM ex)
  GROUP BY id HAVING COUNT(DISTINCT link) > 1
),
-- held with its whole group
cg AS (
  SELECT DISTINCT u.link, u.group_work_id FROM u JOIN cc ON cc.id = u.loser_work_id OR cc.id = u.winner_work_id
)
SELECT u.link, u.group_work_id, u.winner_work_id, u.loser_work_id, u.n_pairs, u.signals,
       COALESCE(u.hold_reason, CASE WHEN cg.group_work_id IS NOT NULL THEN 'cross_class' END) AS hold_reason, u.run_ts
FROM u LEFT JOIN cg ON cg.link = u.link AND cg.group_work_id = u.group_work_id

In [ ]:
SELECT link, hold_reason, COUNT(*) AS losers, COUNT(DISTINCT winner_work_id) AS winners
FROM openalex.works.declared_link_losers GROUP BY 1, 2 ORDER BY 1, 3 DESC

## Tripwire and invariants (before any write)

In [ ]:
WITH x AS (SELECT * FROM openalex.works.declared_link_losers WHERE hold_reason IS NULL),
s AS (SELECT COUNT(*) AS n, COUNT(DISTINCT loser_work_id) AS n_losers,
             COUNT_IF(link = 'preprint') AS n_preprint, COUNT_IF(link = 'version') AS n_version FROM x),
-- version (oxjob #1696): its first night carries the backlog (119,129 executable losers in the 2026-11-23 dry run), later
-- nights only new versions
v AS (SELECT CASE WHEN COUNT_IF(link = 'version') = 0 THEN 150000 ELSE 50000 END AS version_cap
      FROM openalex.works.declared_link_merges),
c AS (SELECT COUNT(*) AS loser_is_winner FROM x JOIN (SELECT DISTINCT winner_work_id FROM x) w ON w.winner_work_id = x.loser_work_id)
SELECT CASE
  WHEN s.n_preprint > 100000 THEN RAISE_ERROR(concat('MergeDeclaredLinks: ', s.n_preprint, ' executable preprint losers > 100,000 tripwire; nothing written'))
  WHEN s.n_version > v.version_cap THEN RAISE_ERROR(concat('MergeDeclaredLinks: ', s.n_version, ' executable version losers > ', v.version_cap, ' tripwire; nothing written'))
  WHEN s.n_losers <> s.n THEN RAISE_ERROR('MergeDeclaredLinks: a loser has two winners')
  WHEN c.loser_is_winner > 0 THEN RAISE_ERROR('MergeDeclaredLinks: a loser is also a winner')
END AS tripwire
FROM s CROSS JOIN c CROSS JOIN v

## Execute, once for all classes
Audit first, aliases before the unpin (the aliases are read from the losers' pinned records), the redirect
(`merged_work_ids`) before the unpin (oxjob #2006), map rows before pins.

In [ ]:
INSERT INTO openalex.works.declared_link_merges_audit
SELECT x.run_ts, x.link, 'pin', x.loser_work_id, x.winner_work_id,
       r.provenance, r.native_id_namespace, r.native_id, r.work_id_source, r.openalex_created_dt, r.openalex_updated_dt, r.seeded_dt, r.seeded_from,
       NULL, NULL, NULL, NULL, NULL, NULL
FROM openalex.works.declared_link_losers x
JOIN openalex.works.location_work_ids r ON r.work_id = x.loser_work_id
WHERE x.hold_reason IS NULL

In [ ]:
INSERT INTO openalex.works.declared_link_merges_audit
SELECT x.run_ts, x.link, 'map', x.loser_work_id, x.winner_work_id,
       NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
       m.doi, m.pmid, m.arxiv, m.title_author, m.created_date, m.updated_date
FROM openalex.works.declared_link_losers x
JOIN openalex.works.work_id_map m ON m.id = x.loser_work_id
WHERE x.hold_reason IS NULL

In [ ]:
CREATE OR REPLACE TABLE openalex.works.declared_link_merge_aliases AS
SELECT DISTINCT x.run_ts, x.link, x.loser_work_id, x.winner_work_id AS id, NULLIF(l.merge_key.doi, '') AS doi, l.merge_key.pmid AS pmid,
       l.merge_key.arxiv AS arxiv, l.merge_key.title_author AS title_author
FROM openalex.works.declared_link_losers x
JOIN openalex.works.location_work_ids r ON r.work_id = x.loser_work_id
JOIN openalex.works.locations_w_types l
  ON l.provenance = r.provenance AND l.native_id_namespace = r.native_id_namespace AND l.native_id = r.native_id
WHERE x.hold_reason IS NULL
  AND COALESCE(NULLIF(l.merge_key.doi, ''), l.merge_key.pmid, l.merge_key.arxiv, l.merge_key.title_author) IS NOT NULL

In [ ]:
INSERT INTO openalex.works.work_id_map (id, doi, pmid, arxiv, title_author, created_date, updated_date)
SELECT DISTINCT id, doi, pmid, arxiv, title_author, current_date(), current_timestamp()
FROM openalex.works.declared_link_merge_aliases x
WHERE NOT EXISTS (SELECT 1 FROM openalex.works.work_id_map m WHERE m.id = x.id AND m.doi <=> x.doi AND m.pmid <=> x.pmid
                  AND m.arxiv <=> x.arxiv AND m.title_author <=> x.title_author)

In [ ]:
INSERT INTO openalex.works.declared_link_merges_audit
SELECT run_ts, link, 'alias', loser_work_id, id, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
       doi, pmid, arxiv, title_author, current_date(), NULL
FROM openalex.works.declared_link_merge_aliases

In [ ]:
-- oxjob #2006: the redirect goes in before the unpin (the next two cells). A failure between them now leaves a live
-- loser with a redirect, which merged_work_ids_map leaves out (live wins) and the next run completes; before, it left
-- an unpinned loser with no redirect, served as deleted (#1950).
MERGE INTO openalex.works.merged_work_ids m
USING (SELECT DISTINCT loser_work_id, winner_work_id, run_ts,
              CASE link WHEN 'preprint' THEN 'preprint_link_nightly' WHEN 'version' THEN 'crossref_version_nightly' END AS source
       FROM openalex.works.declared_link_losers WHERE hold_reason IS NULL) s
ON m.loser_work_id = s.loser_work_id AND m.winner_work_id = s.winner_work_id
WHEN NOT MATCHED THEN INSERT (loser_work_id, winner_work_id, merged_at, source)
VALUES (s.loser_work_id, s.winner_work_id, s.run_ts, s.source)

In [ ]:
DELETE FROM openalex.works.work_id_map m
WHERE EXISTS (SELECT 1 FROM openalex.works.declared_link_losers x WHERE x.hold_reason IS NULL AND x.loser_work_id = m.id)

In [ ]:
DELETE FROM openalex.works.location_work_ids r
WHERE EXISTS (SELECT 1 FROM openalex.works.declared_link_losers x WHERE x.hold_reason IS NULL AND x.loser_work_id = r.work_id)

In [ ]:
INSERT INTO openalex.works.declared_link_merges
SELECT run_ts, link, loser_work_id, winner_work_id, group_work_id, n_pairs, signals
FROM openalex.works.declared_link_losers WHERE hold_reason IS NULL

## Citations follow the merge
Rows citing a ledgered loser move to its winner, and the loser's own reference rows come along (they keep their
`native_id` + `ref_ind`, so they never collide with the winner's). Idempotent: only rows still keyed to a loser move.

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW declared_link_repoint AS
SELECT p.loser_work_id, p.winner_work_id
FROM (SELECT loser_work_id, MIN(winner_work_id) AS winner_work_id
      FROM (SELECT loser_work_id, winner_work_id FROM openalex.works.preprint_link_merges
            UNION ALL
            SELECT loser_work_id, winner_work_id FROM openalex.works.declared_link_merges)
      GROUP BY loser_work_id) p
WHERE NOT EXISTS (SELECT 1 FROM openalex.works.location_work_ids r WHERE r.work_id = p.loser_work_id)

In [ ]:
INSERT INTO openalex.works.declared_link_merges_refs_audit
SELECT current_timestamp(), 'cited', r.citing_work_id, r.native_id, r.native_id_namespace, r.ref_ind, r.cited_work_id, p.winner_work_id
FROM openalex.works.work_references r JOIN declared_link_repoint p ON r.cited_work_id = p.loser_work_id
UNION ALL
SELECT current_timestamp(), 'citing', r.citing_work_id, r.native_id, r.native_id_namespace, r.ref_ind, r.citing_work_id, p.winner_work_id
FROM openalex.works.work_references r JOIN declared_link_repoint p ON r.citing_work_id = p.loser_work_id

In [ ]:
MERGE INTO openalex.works.work_references r
USING declared_link_repoint p ON r.cited_work_id = p.loser_work_id
WHEN MATCHED THEN UPDATE SET r.cited_work_id = p.winner_work_id, r.updated_timestamp = current_timestamp()

In [ ]:
MERGE INTO openalex.works.work_references r
USING declared_link_repoint p ON r.citing_work_id = p.loser_work_id
WHEN MATCHED THEN UPDATE SET r.citing_work_id = p.winner_work_id, r.updated_timestamp = current_timestamp()

In [ ]:
WITH x AS (
  SELECT link, COUNT(*) AS losers_tonight, COUNT_IF(hold_reason IS NULL) AS losers_merged,
         COUNT(DISTINCT CASE WHEN hold_reason IS NULL THEN winner_work_id END) AS winners
  FROM openalex.works.declared_link_losers GROUP BY link
),
a AS (SELECT link, COUNT(*) AS alias_rows FROM openalex.works.declared_link_merge_aliases GROUP BY link),
h AS (SELECT link, COUNT(*) AS losers_merged_all_time FROM openalex.works.declared_link_merges GROUP BY link)
SELECT x.link, x.losers_tonight, x.losers_merged, x.winners, COALESCE(a.alias_rows, 0) AS alias_rows,
       COALESCE(h.losers_merged_all_time, 0) AS losers_merged_all_time
FROM x LEFT JOIN a ON a.link = x.link LEFT JOIN h ON h.link = x.link ORDER BY x.link